In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from vocal_assistant.vocal_assistant import VocalAssistant
from vocal_assistant.emotion.emotion_model import process_func, EmotionModel
from transformers import Wav2Vec2Processor
from vocal_assistant.emotion.predict_emotion import load_trained_model, predict_emotion

In [2]:
device = 'cpu'

audeering_model_name = 'audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim'
audeering_processor = Wav2Vec2Processor.from_pretrained(audeering_model_name)
audeering_model = EmotionModel.from_pretrained(audeering_model_name).to(device)

custom_model_name = "./vocal_assistant/emotion/model_checkpoint_sampled.pth"
custom_model, custom_processor = load_trained_model(custom_model_name)

/Users/lucabellani/Documents/UNI/Tesi/Recommersion/vocal_assistant/emotion/predict_emotion.py:149: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(chec

Loaded trained model from checkpoint.


In [3]:
import pandas as pd

songs = pd.read_pickle("./data/Songs")

In [4]:
vc = VocalAssistant(1)
vc.talk("What is your mood today?")
while True:
    command, vocal_file = vc.take_command()
    print(command)
    break

print("Audeering: ")
audeering = process_func(vocal_file, 16000)[0]
print(audeering)
print("Custom: ")
custom = list(predict_emotion(custom_model, custom_processor, vocal_file).values())
print(custom)
#custom model seems to give the same results: overfitting?
# [0.4621863067150116, 0.5704010128974915, 0.615027904510498]


  listening....
Sample rate: 16000
Numpy array shape: (50156,)
i am sad
Audeering: 
[-0.14537978  0.02989404 -0.04063611]
Custom: 
[0.5390353202819824, 0.5625609159469604, 0.6179229617118835]


In [5]:
import numpy as np
dim_vec = np.array(custom[0:2])
songs_list = pd.DataFrame({"id": songs["musicId"], "eucl_dist":songs[["Valence", "Arousal"]]\
                           .apply(lambda x: np.linalg.norm(x - dim_vec), axis=1), "Valence": songs["Valence"], "Arousal": songs["Arousal"],\
                            "title":songs["title"], "artist": songs["artist"], "mp3_file":songs["mp3_file"]})

songs_list = songs_list.sort_values(by="eucl_dist")[:5]
songs_list

#Careful that in the dataset there are some duplicates
#In a Deam Metadata file there titles with \t

,id,eucl_dist,Valence,Arousal,title,artist,mp3_file
691,1921,0.006791,0.544444,0.566667,\tSambarama\t,Juanitos\t,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
160,1192,0.007026,0.533333,0.566667,\tLucky\t,Steven Arntson\t,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
214,283,0.012533,0.537500,0.575000,U With Me?,Drake,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
144,177,0.012654,0.537500,0.550000,For Free,DJ Khaled,"[0.0, 5.639831e-14, 2.7850728e-13, 2.507313e-1..."
537,1695,0.016150,0.544444,0.577778,\tTrack 6\t,Sunny Jain's Red Baraat Festival\t,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [6]:
import sounddevice as sd

for i in range(len(songs_list)):
    sd.play(songs_list["mp3_file"].iloc[i], 44100)
    sd.wait()
